In [ ]:
# MCP SCENARIO: “Smart IT Helpdesk Assistant”
# 🧩 Scenario Background

# You are working in a company called ABC Corp.

# Employees face issues like:

# VPN not working
# Printer not responding
# Software errors

# 👉 Instead of calling IT support, employees use an AI Helpdesk Bot.

# 🤖 What this Bot Should Do

# When a user types a problem:

# Understand the issue
# Decide if a ticket is needed
# Identify:
# Category (Network / Hardware / General)
# Priority (High / Medium)
# Create a ticket
# Show confirmation
# 🧠 How MCP Fits Here
# Component	Role in Scenario
# Agent	Helpdesk Bot
# MCP Layer	Decision + Tool calling
# Tool	Ticket Creation System
# User	Employee




# ============================================
# STEP 0: DATABASE (Simulated storage)
# ============================================

tickets_db = []  # This stores all tickets


# ============================================
# STEP 1: TOOL (MCP TOOL)
# ============================================

def create_ticket(issue, priority, category):
    """
    This function simulates a TOOL in MCP
    In real world → API / Database / ServiceNow
    """

    ticket_id = f"INC{1000 + len(tickets_db)}"

    ticket = {
        "ticket_id": ticket_id,
        "issue": issue,
        "priority": priority,
        "category": category
    }

    tickets_db.append(ticket)

    return ticket


# ============================================
# STEP 2: AGENT REASONING (LLM SIMULATION)
# ============================================

def analyze_input(user_input):
    """
    Simulates how an LLM understands user input
    Extracts:
    - category
    - priority
    """

    text = user_input.lower()

    # 🔹 Category Detection
    if "vpn" in text:
        category = "network"
    elif "printer" in text:
        category = "hardware"
    elif "email" in text:
        category = "software"
    else:
        category = "general"

    # 🔹 Priority Detection
    if "urgent" in text or "immediately" in text:
        priority = "high"
    elif "slow" in text:
        priority = "low"
    else:
        priority = "medium"

    return category, priority


# ============================================
# STEP 3: DECISION ENGINE (MCP CORE)
# ============================================

def should_call_tool(user_input):
    """
    Decides whether to call a tool or not
    This is MCP decision layer
    """

    keywords = ["issue", "problem", "ticket", "not working"]

    return any(word in user_input.lower() for word in keywords)


# ============================================
# STEP 4: MCP ORCHESTRATOR
# ============================================

def mcp_agent(user_input):
    """
    This is the MAIN MCP FLOW
    It connects:
    Agent → Decision → Tool → Response
    """

    print("\n🧠 Agent received input:", user_input)

    # STEP 4.1: Decision
    if should_call_tool(user_input):

        print("➡️ Decision: Tool call required")

        # STEP 4.2: Analyze input
        category, priority = analyze_input(user_input)

        print(f"📊 Extracted → Category: {category}, Priority: {priority}")

        # STEP 4.3: Prepare payload (MCP format)
        payload = {
            "issue": user_input,
            "priority": priority,
            "category": category
        }

        print("📦 MCP Payload:", payload)

        # STEP 4.4: Call tool
        result = create_ticket(**payload)

        print("⚙️ Tool executed successfully")

        # STEP 4.5: Final response
        return f"""
        ✅ Ticket Created Successfully!

        Ticket ID: {result['ticket_id']}
        Issue: {result['issue']}
        Category: {result['category']}
        Priority: {result['priority']}
        """

    else:
        print("➡️ Decision: No tool needed (AI response)")

        return "🤖 AI Response: Please describe your issue clearly."


# ============================================
# STEP 5: RUN INTERACTIVE LOOP
# ============================================

print("🚀 MCP Demo Started (Type 'exit' to stop)\n")

while True:

    user_input = input("Enter your query: ")

    if user_input.lower() == "exit":
        print("👋 Exiting MCP demo...")
        break

    response = mcp_agent(user_input)
    print(response)


🚀 MCP Demo Started (Type 'exit' to stop)

Enter your query: My email is not working properly

🧠 Agent received input: My email is not working properly
➡️ Decision: Tool call required
📊 Extracted → Category: software, Priority: medium
📦 MCP Payload: {'issue': 'My email is not working properly', 'priority': 'medium', 'category': 'software'}
⚙️ Tool executed successfully

        ✅ Ticket Created Successfully!

        Ticket ID: INC1000
        Issue: My email is not working properly
        Category: software
        Priority: medium
        
Enter your query: my vpn is not working fix it immediately

🧠 Agent received input: my vpn is not working fix it immediately
➡️ Decision: Tool call required
📊 Extracted → Category: network, Priority: high
📦 MCP Payload: {'issue': 'my vpn is not working fix it immediately', 'priority': 'high', 'category': 'network'}
⚙️ Tool executed successfully

        ✅ Ticket Created Successfully!

        Ticket ID: INC1001
        Issue: my vpn is not working 

In [ ]:
# ============================================
# MCP SMART IT HELPDESK (WITH GROQ - FIXED)
# ============================================

from groq import Groq

# ============================================
# STEP 0: API KEY INPUT
# ============================================

groq_api_key = input("🔑 Enter your Groq API Key: ").strip()
client = Groq(api_key=groq_api_key)

# ============================================
# STEP 1: DATABASE
# ============================================

tickets_db = []

# ============================================
# STEP 2: TOOL
# ============================================

def create_ticket(issue, priority, category):
    ticket_id = f"INC{1000 + len(tickets_db)}"

    ticket = {
        "ticket_id": ticket_id,
        "issue": issue,
        "priority": priority,
        "category": category
    }

    tickets_db.append(ticket)
    return ticket

# ============================================
# STEP 3: LLM ANALYSIS (UPDATED MODEL)
# ============================================

def analyze_input_llm(user_input):

    prompt = f"""
You are an IT helpdesk classifier.

Classify the issue into:
- Category: network / hardware / software / general
- Priority: high / medium / low

Rules:
- urgent/immediately → high
- slow → low
- otherwise → medium

Input: "{user_input}"

Output ONLY in this format:
category: <value>
priority: <value>
"""

    try:
        response = client.chat.completions.create(
            model="llama-3.1-70b-versatile",   # ✅ FIXED MODEL
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )

        result = response.choices[0].message.content.lower()

        category = "general"
        priority = "medium"

        for line in result.split("\n"):
            if "category" in line:
                category = line.split(":")[-1].strip()
            elif "priority" in line:
                priority = line.split(":")[-1].strip()

        return category, priority

    except Exception as e:
        print("⚠️ LLM Error:", e)
        print("⚠️ Falling back to rule-based system")

        return analyze_input_fallback(user_input)


# ============================================
# STEP 3B: FALLBACK (VERY IMPORTANT 🔥)
# ============================================

def analyze_input_fallback(user_input):

    text = user_input.lower()

    if "vpn" in text or "network" in text or "internet" in text:
        category = "network"
    elif "printer" in text or "mouse" in text or "keyboard" in text:
        category = "hardware"
    elif "email" in text or "software" in text:
        category = "software"
    else:
        category = "general"

    if "urgent" in text or "immediately" in text:
        priority = "high"
    elif "slow" in text:
        priority = "low"
    else:
        priority = "medium"

    return category, priority


# ============================================
# STEP 4: DECISION ENGINE (IMPROVED)
# ============================================

def should_call_tool(user_input):

    keywords = [
        "issue", "problem", "ticket", "not working",
        "error", "fail", "down", "unable", "can't"
    ]

    return any(word in user_input.lower() for word in keywords)


# ============================================
# STEP 5: MCP AGENT
# ============================================

def mcp_agent(user_input):

    print("\n🧠 Agent received input:", user_input)

    if should_call_tool(user_input):

        print("➡️ Decision: Tool call required")

        # 🔥 LLM + fallback system
        category, priority = analyze_input_llm(user_input)

        print(f"📊 Extracted → Category: {category}, Priority: {priority}")

        payload = {
            "issue": user_input,
            "priority": priority,
            "category": category
        }

        print("📦 MCP Payload:", payload)

        result = create_ticket(**payload)

        print("⚙️ Tool executed successfully")

        return f"""
✅ Ticket Created Successfully!

Ticket ID: {result['ticket_id']}
Issue: {result['issue']}
Category: {result['category']}
Priority: {result['priority']}
"""

    else:
        print("➡️ Decision: No tool needed (AI response)")

        try:
            response = client.chat.completions.create(
                model="llama-3.1-8b-instant",   # ✅ FIXED MODEL
                messages=[{"role": "user", "content": user_input}]
            )

            return response.choices[0].message.content

        except Exception as e:
            return "🤖 Please describe your issue clearly."


# ============================================
# STEP 6: RUN LOOP
# ============================================

print("\n🚀 MCP + Groq Helpdesk Started (type 'exit' to stop)\n")

while True:

    user_input = input("Enter your query: ")

    if user_input.lower() == "exit":
        print("👋 Exiting MCP system...")
        break

    response = mcp_agent(user_input)
    print(response)

🔑 Enter your Groq API Key: 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

🚀 MCP + Groq Helpdesk Started (type 'exit' to stop)

Enter your query: my office vpn is not working properly , fix it urgently

🧠 Agent received input: my office vpn is not working properly , fix it urgently
➡️ Decision: Tool call required
⚠️ LLM Error: Error code: 400 - {'error': {'message': 'The model `llama-3.1-70b-versatile` has been decommissioned and is no longer supported. Please refer to https://console.groq.com/docs/deprecations for a recommendation on which model to use instead.', 'type': 'invalid_request_error', 'code': 'model_decommissioned'}}
⚠️ Falling back to rule-based system
📊 Extracted → Category: network, Priority: high
📦 MCP Payload: {'issue': 'my office vpn is not working properly , fix it urgently', 'priority': 'high', 'category': 'network'}
⚙️ Tool executed successfully

✅ Ticket Created Successfully!

Ticket ID: INC1000
Issue: my office vpn is not working properly , fix it urgen

In [ ]:
models = client.models.list()

for m in models.data:
    print(m.id)

openai/gpt-oss-20b
qwen/qwen3-32b
meta-llama/llama-prompt-guard-2-86m
meta-llama/llama-4-scout-17b-16e-instruct
openai/gpt-oss-120b
meta-llama/llama-prompt-guard-2-22m
groq/compound-mini
llama-3.3-70b-versatile
canopylabs/orpheus-v1-english
whisper-large-v3-turbo
groq/compound
llama-3.1-8b-instant
moonshotai/kimi-k2-instruct-0905
openai/gpt-oss-safeguard-20b
canopylabs/orpheus-arabic-saudi
moonshotai/kimi-k2-instruct
allam-2-7b
whisper-large-v3


# MCP SCENARIO: “Smart HR Onboarding Assistant”
🧩 Scenario Background
You are working in a company called XYZ Corp.
New employees often face challenges during onboarding, such as:
- Trouble accessing payroll portal
- Confusion about leave policies
- Difficulty setting up email accounts
- Questions about training schedules
👉 Instead of emailing HR or waiting for responses, employees use an AI Onboarding Bot.

🤖 What this Bot Should Do
When a new hire types a question/problem:
- Understand the query (e.g., “I can’t log into payroll”)
- Decide if escalation to HR is needed
- Identify:
- Category (Payroll / Policy / IT Setup / Training)
- Priority (High / Medium)
- Create a support ticket if required
- Provide instant guidance (FAQs, step-by-step instructions)
- Show confirmation and next steps

🧠 How MCP Fits Here
|  |  |
|  |  |
|  |  |
|  |  |
|  |  |



This way, the MCP framework is reused in a Human Resources context, where the AI assistant streamlines onboarding, reduces HR workload, and ensures employees feel supported from day one.
Would you like me to design another variation in a customer service setting (like retail or banking), so you can compare how MCP adapts across industries?

In [ ]:
# ============================================
# MCP SMART HR ONBOARDING ASSISTANT
# ============================================

from groq import Groq

# ============================================
# STEP 0: API KEY
# ============================================

groq_api_key = input("🔑 Enter your Groq API Key: ").strip()
client = Groq(api_key=groq_api_key)

# ============================================
# STEP 1: DATABASE
# ============================================

tickets_db = []

# ============================================
# STEP 2: TOOL (HR TICKET SYSTEM)
# ============================================

def create_hr_ticket(issue, priority, category):
    ticket_id = f"HR{2000 + len(tickets_db)}"

    ticket = {
        "ticket_id": ticket_id,
        "issue": issue,
        "priority": priority,
        "category": category
    }

    tickets_db.append(ticket)
    return ticket

# ============================================
# STEP 3: FAQ RESPONSES (INSTANT HELP)
# ============================================

def get_faq_response(category):

    faqs = {
        "payroll": "💰 Payroll Portal: Visit portal.xyz.com → Click 'Reset Password' if login fails.",
        "policy": "📄 Leave Policy: You get 20 paid leaves/year. Apply via HRMS portal.",
        "it setup": "💻 Email Setup: Use Outlook → login with company email → follow setup guide.",
        "training": "📚 Training: Check LMS portal → 'My Courses' → onboarding modules."
    }

    return faqs.get(category, None)

# ============================================
# STEP 4: LLM ANALYSIS
# ============================================

def analyze_input_llm(user_input):

    prompt = f"""
You are an HR onboarding assistant.

Classify the query into:
Category: payroll / policy / it setup / training
Priority: high / medium

Rules:
- login issues, access problems → high
- general questions → medium

Input: "{user_input}"

Output ONLY in this format:
category: <value>
priority: <value>
"""

    try:
        response = client.chat.completions.create(
            model="llama-3.1-70b-versatile",
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )

        result = response.choices[0].message.content.lower()

        category = "policy"
        priority = "medium"

        for line in result.split("\n"):
            if "category" in line:
                category = line.split(":")[-1].strip()
            elif "priority" in line:
                priority = line.split(":")[-1].strip()

        return category, priority

    except Exception as e:
        print("⚠️ LLM Error:", e)
        return analyze_input_fallback(user_input)


# ============================================
# STEP 4B: FALLBACK
# ============================================

def analyze_input_fallback(user_input):

    text = user_input.lower()

    if "payroll" in text or "salary" in text:
        category = "payroll"
    elif "leave" in text or "policy" in text:
        category = "policy"
    elif "email" in text or "setup" in text:
        category = "it setup"
    elif "training" in text:
        category = "training"
    else:
        category = "policy"

    if "can't" in text or "unable" in text or "not working" in text:
        priority = "high"
    else:
        priority = "medium"

    return category, priority


# ============================================
# STEP 5: DECISION ENGINE (SMART 🔥)
# ============================================

def should_create_ticket(user_input):

    keywords = ["can't", "unable", "not working", "error", "issue", "problem"]

    return any(word in user_input.lower() for word in keywords)


# ============================================
# STEP 6: MCP AGENT
# ============================================

def hr_mcp_agent(user_input):

    print("\n🧠 HR Agent received input:", user_input)

    category, priority = analyze_input_llm(user_input)

    print(f"📊 Extracted → Category: {category}, Priority: {priority}")

    # 🔥 FIRST: Try FAQ response
    faq = get_faq_response(category)

    if faq:
        print("📘 Providing instant guidance")

    # 🔥 DECISION: Ticket or not
    if should_create_ticket(user_input):

        print("➡️ Decision: Create HR ticket")

        payload = {
            "issue": user_input,
            "priority": priority,
            "category": category
        }

        result = create_hr_ticket(**payload)

        print("⚙️ HR Ticket created")

        return f"""
{faq if faq else ""}

✅ HR Ticket Created!

Ticket ID: {result['ticket_id']}
Category: {result['category']}
Priority: {result['priority']}

📌 HR team will contact you shortly.
"""

    else:
        print("➡️ Decision: FAQ response only")

        if faq:
            return faq
        else:
            return "🤖 Please provide more details about your query."


# ============================================
# STEP 7: RUN LOOP
# ============================================

print("\n🚀 HR Onboarding MCP Started (type 'exit' to stop)\n")

while True:

    user_input = input("Enter your query: ")

    if user_input.lower() == "exit":
        print("👋 Exiting HR assistant...")
        break

    response = hr_mcp_agent(user_input)
    print(response)

🔑 Enter your Groq API Key: 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

🚀 HR Onboarding MCP Started (type 'exit' to stop)

Enter your query: What is the leave policy?

🧠 HR Agent received input: What is the leave policy?
⚠️ LLM Error: Error code: 400 - {'error': {'message': 'The model `llama-3.1-70b-versatile` has been decommissioned and is no longer supported. Please refer to https://console.groq.com/docs/deprecations for a recommendation on which model to use instead.', 'type': 'invalid_request_error', 'code': 'model_decommissioned'}}
📊 Extracted → Category: policy, Priority: medium
📘 Providing instant guidance
➡️ Decision: FAQ response only
📄 Leave Policy: You get 20 paid leaves/year. Apply via HRMS portal.
Enter your query: How do I set up my company email?

🧠 HR Agent received input: How do I set up my company email?
⚠️ LLM Error: Error code: 400 - {'error': {'message': 'The model `llama-3.1-70b-versatile` has been decommissioned and is no longer supported. Please ref

In [ ]:
# ============================================
# MCP SMART HR ONBOARDING (NO LLM)
# ============================================

# ============================================
# STEP 0: DATABASE
# ============================================

tickets_db = []

# ============================================
# STEP 1: TOOL (HR TICKET SYSTEM)
# ============================================

def create_hr_ticket(issue, priority, category):

    ticket_id = f"HR{2000 + len(tickets_db)}"

    ticket = {
        "ticket_id": ticket_id,
        "issue": issue,
        "priority": priority,
        "category": category
    }

    tickets_db.append(ticket)

    return ticket

# ============================================
# STEP 2: FAQ RESPONSES
# ============================================

def get_faq_response(category):

    faqs = {
        "payroll": "💰 Payroll Portal: Visit portal.xyz.com → Click 'Reset Password' if login fails.",
        "policy": "📄 Leave Policy: You get 20 paid leaves/year. Apply via HRMS portal.",
        "it setup": "💻 Email Setup: Use Outlook → login with company email → follow setup guide.",
        "training": "📚 Training: Go to LMS portal → 'My Courses' → complete onboarding modules."
    }

    return faqs.get(category, None)

# ============================================
# STEP 3: INPUT ANALYSIS (RULE-BASED)
# ============================================

def analyze_input(user_input):

    text = user_input.lower()

    # 🔹 CATEGORY DETECTION
    if "payroll" in text or "salary" in text:
        category = "payroll"

    elif "leave" in text or "policy" in text:
        category = "policy"

    elif "email" in text or "setup" in text or "login" in text:
        category = "it setup"

    elif "training" in text or "course" in text:
        category = "training"

    else:
        category = "policy"

    # 🔹 PRIORITY DETECTION
    if "urgent" in text or "immediately" in text:
        priority = "high"

    elif "can't" in text or "unable" in text or "not working" in text:
        priority = "high"

    else:
        priority = "medium"

    return category, priority

# ============================================
# STEP 4: DECISION ENGINE (MCP CORE)
# ============================================

def should_create_ticket(user_input):

    keywords = ["issue", "problem", "error", "not working", "can't", "unable"]

    return any(word in user_input.lower() for word in keywords)

# ============================================
# STEP 5: MCP AGENT
# ============================================

def hr_mcp_agent(user_input):

    print("\n🧠 HR Agent received input:", user_input)

    # STEP 5.1: Analyze input
    category, priority = analyze_input(user_input)

    print(f"📊 Extracted → Category: {category}, Priority: {priority}")

    # STEP 5.2: Get FAQ
    faq = get_faq_response(category)

    # STEP 5.3: Decision
    if should_create_ticket(user_input):

        print("➡️ Decision: Create HR ticket")

        payload = {
            "issue": user_input,
            "priority": priority,
            "category": category
        }

        print("📦 MCP Payload:", payload)

        result = create_hr_ticket(**payload)

        print("⚙️ Ticket created")

        return f"""
{faq if faq else ""}

✅ HR Ticket Created!

Ticket ID: {result['ticket_id']}
Category: {result['category']}
Priority: {result['priority']}

📌 HR team will contact you soon.
"""

    else:
        print("➡️ Decision: FAQ response only")

        if faq:
            return faq
        else:
            return "🤖 Please provide more details."

# ============================================
# STEP 6: RUN LOOP
# ============================================

print("🚀 HR MCP (No LLM) Started (type 'exit' to stop)\n")

while True:

    user_input = input("Enter your query: ")

    if user_input.lower() == "exit":
        print("👋 Exiting HR assistant...")
        break

    response = hr_mcp_agent(user_input)
    print(response)

🚀 HR MCP (No LLM) Started (type 'exit' to stop)

Enter your query: I have a problem with payroll

🧠 HR Agent received input: I have a problem with payroll
📊 Extracted → Category: payroll, Priority: medium
➡️ Decision: Create HR ticket
📦 MCP Payload: {'issue': 'I have a problem with payroll', 'priority': 'medium', 'category': 'payroll'}
⚙️ Ticket created

💰 Payroll Portal: Visit portal.xyz.com → Click 'Reset Password' if login fails.

✅ HR Ticket Created!

Ticket ID: HR2000
Category: payroll
Priority: medium

📌 HR team will contact you soon.

Enter your query: How to setup email?

🧠 HR Agent received input: How to setup email?
📊 Extracted → Category: it setup, Priority: medium
➡️ Decision: FAQ response only
💻 Email Setup: Use Outlook → login with company email → follow setup guide.
Enter your query: exit
👋 Exiting HR assistant...


# MCP SCENARIO: “Smart Banking Support Assistant”
🧩 Scenario Background
You are working in a company called FinTrust Bank.
Customers often face issues such as:
- Credit card not working
- Trouble with online banking login
- Queries about loan status
- Transaction disputes
👉 Instead of calling customer care, customers use an AI Banking Support Bot.

🤖 What this Bot Should Do
When a customer types a problem:
- Understand the issue (e.g., “My card was declined”)
- Decide if escalation to a human agent is needed
- Identify:
- Category (Card Services / Online Banking / Loans / Transactions)
- Priority (High / Medium)
- Create a support ticket if required
- Provide instant guidance (FAQs, troubleshooting steps, policy info)
- Show confirmation and next steps

🧠 How MCP Fits Here
|  |  |
|  |  |
|  |  |
|  |  |
|  |  |



This way, MCP is applied in a financial services context, where the AI assistant reduces call center load, provides quick resolutions, and ensures customers feel supported with secure, reliable guidance.
Would you like me to craft one more in a healthcare setting (like hospital patient support), so you can see how MCP adapts to critical service environments?

In [ ]:
# ============================================
# MCP SMART BANKING SUPPORT (NO LLM)
# ============================================

# ============================================
# STEP 0: DATABASE
# ============================================

tickets_db = []

# ============================================
# STEP 1: TOOL (BANKING TICKET SYSTEM)
# ============================================

def create_ticket(issue, priority, category):

    ticket_id = f"BNK{3000 + len(tickets_db)}"

    ticket = {
        "ticket_id": ticket_id,
        "issue": issue,
        "priority": priority,
        "category": category
    }

    tickets_db.append(ticket)

    return ticket

# ============================================
# STEP 2: FAQ RESPONSES
# ============================================

def get_faq_response(category):

    faqs = {
        "card services": "💳 Card Help: Check if card is active → verify PIN → ensure sufficient balance.",
        "online banking": "🌐 Login Help: Reset password via 'Forgot Password' → ensure correct credentials.",
        "loans": "🏦 Loan Status: Visit loan portal → enter application ID → check status.",
        "transactions": "💰 Transaction Issues: Check recent transactions → allow 24 hrs → contact support if unresolved."
    }

    return faqs.get(category, None)

# ============================================
# STEP 3: INPUT ANALYSIS (RULE-BASED)
# ============================================

def analyze_input(user_input):

    text = user_input.lower()

    # 🔹 CATEGORY DETECTION
    if "card" in text or "credit" in text or "debit" in text:
        category = "card services"

    elif "login" in text or "password" in text or "banking" in text:
        category = "online banking"

    elif "loan" in text:
        category = "loans"

    elif "transaction" in text or "payment" in text or "debited" in text:
        category = "transactions"

    else:
        category = "transactions"

    # 🔹 PRIORITY DETECTION
    if "fraud" in text or "unauthorized" in text or "urgent" in text:
        priority = "high"

    elif "not working" in text or "failed" in text or "declined" in text:
        priority = "high"

    else:
        priority = "medium"

    return category, priority

# ============================================
# STEP 4: DECISION ENGINE (MCP CORE)
# ============================================

def should_create_ticket(user_input):

    keywords = ["issue", "problem", "failed", "declined", "unauthorized", "fraud", "not working"]

    return any(word in user_input.lower() for word in keywords)

# ============================================
# STEP 5: MCP AGENT
# ============================================

def banking_mcp_agent(user_input):

    print("\n🧠 Banking Agent received input:", user_input)

    # STEP 5.1: Analyze
    category, priority = analyze_input(user_input)

    print(f"📊 Extracted → Category: {category}, Priority: {priority}")

    # STEP 5.2: FAQ
    faq = get_faq_response(category)

    # STEP 5.3: Decision
    if should_create_ticket(user_input):

        print("➡️ Decision: Create support ticket")

        payload = {
            "issue": user_input,
            "priority": priority,
            "category": category
        }

        print("📦 MCP Payload:", payload)

        result = create_ticket(**payload)

        print("⚙️ Ticket created")

        return f"""
{faq if faq else ""}

✅ Support Ticket Created!

Ticket ID: {result['ticket_id']}
Category: {result['category']}
Priority: {result['priority']}

📌 Our banking team will contact you shortly.
"""

    else:
        print("➡️ Decision: FAQ response only")

        if faq:
            return faq
        else:
            return "🤖 Please provide more details."

# ============================================
# STEP 6: RUN LOOP
# ============================================

print("🚀 Banking MCP Started (type 'exit' to stop)\n")

while True:

    user_input = input("Enter your query: ")

    if user_input.lower() == "exit":
        print("👋 Exiting Banking Assistant...")
        break

    response = banking_mcp_agent(user_input)
    print(response)

🚀 Banking MCP Started (type 'exit' to stop)

Enter your query: My credit card was declined

🧠 Banking Agent received input: My credit card was declined
📊 Extracted → Category: card services, Priority: high
➡️ Decision: Create support ticket
📦 MCP Payload: {'issue': 'My credit card was declined', 'priority': 'high', 'category': 'card services'}
⚙️ Ticket created

💳 Card Help: Check if card is active → verify PIN → ensure sufficient balance.

✅ Support Ticket Created!

Ticket ID: BNK3000
Category: card services
Priority: high

📌 Our banking team will contact you shortly.

Enter your query: how to apply a chechkbook

🧠 Banking Agent received input: how to apply a chechkbook
📊 Extracted → Category: transactions, Priority: medium
➡️ Decision: FAQ response only
💰 Transaction Issues: Check recent transactions → allow 24 hrs → contact support if unresolved.
Enter your query: exit
👋 Exiting Banking Assistant...


In [ ]:
# ============================================
# MCP SMART BANKING SUPPORT (WITH LLM)
# ============================================

from groq import Groq

# ============================================
# STEP 0: API KEY
# ============================================

groq_api_key = input("🔑 Enter your Groq API Key: ").strip()
client = Groq(api_key=groq_api_key)

# ============================================
# STEP 1: DATABASE
# ============================================

tickets_db = []

# ============================================
# STEP 2: TOOL (BANKING TICKET SYSTEM)
# ============================================

def create_ticket(issue, priority, category):

    ticket_id = f"BNK{3000 + len(tickets_db)}"

    ticket = {
        "ticket_id": ticket_id,
        "issue": issue,
        "priority": priority,
        "category": category
    }

    tickets_db.append(ticket)

    return ticket

# ============================================
# STEP 3: FAQ RESPONSES
# ============================================

def get_faq_response(category):

    faqs = {
        "card services": "💳 Check card status → verify PIN → ensure balance.",
        "online banking": "🌐 Reset password using 'Forgot Password'.",
        "loans": "🏦 Check loan status via loan portal.",
        "transactions": "💰 Check transaction history → wait 24 hrs if pending."
    }

    return faqs.get(category, None)

# ============================================
# STEP 4: LLM ANALYSIS
# ============================================

def analyze_input_llm(user_input):

    prompt = f"""
You are a banking support assistant.

Classify the issue into:
Category: card services / online banking / loans / transactions
Priority: high / medium

Rules:
- fraud, unauthorized, money deducted → high
- login failure, card declined → high
- general queries → medium

Input: "{user_input}"

Output ONLY in this format:
category: <value>
priority: <value>
"""

    try:
        response = client.chat.completions.create(
            model="llama-3.1-70b-versatile",
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )

        result = response.choices[0].message.content.lower()

        category = "transactions"
        priority = "medium"

        for line in result.split("\n"):
            if "category" in line:
                category = line.split(":")[-1].strip()
            elif "priority" in line:
                priority = line.split(":")[-1].strip()

        return category, priority

    except Exception as e:
        print("⚠️ LLM Error:", e)
        return analyze_input_fallback(user_input)

# ============================================
# STEP 4B: FALLBACK
# ============================================

def analyze_input_fallback(user_input):

    text = user_input.lower()

    if "card" in text:
        category = "card services"
    elif "login" in text or "password" in text:
        category = "online banking"
    elif "loan" in text:
        category = "loans"
    else:
        category = "transactions"

    if "fraud" in text or "unauthorized" in text or "deducted" in text:
        priority = "high"
    else:
        priority = "medium"

    return category, priority

# ============================================
# STEP 5: DECISION ENGINE
# ============================================

def should_create_ticket(user_input):

    keywords = [
        "problem", "issue", "failed", "declined",
        "unauthorized", "fraud", "not working", "deducted"
    ]

    return any(word in user_input.lower() for word in keywords)

# ============================================
# STEP 6: MCP AGENT
# ============================================

def banking_mcp_agent(user_input):

    print("\n🧠 Banking Agent received input:", user_input)

    # 🔥 LLM understanding
    category, priority = analyze_input_llm(user_input)

    print(f"📊 Extracted → Category: {category}, Priority: {priority}")

    # FAQ
    faq = get_faq_response(category)

    # Decision
    if should_create_ticket(user_input):

        print("➡️ Decision: Create support ticket")

        payload = {
            "issue": user_input,
            "priority": priority,
            "category": category
        }

        print("📦 MCP Payload:", payload)

        result = create_ticket(**payload)

        print("⚙️ Ticket created")

        return f"""
{faq if faq else ""}

✅ Support Ticket Created!

Ticket ID: {result['ticket_id']}
Category: {result['category']}
Priority: {result['priority']}

📌 Our banking team will contact you shortly.
"""

    else:
        print("➡️ Decision: FAQ response only")

        try:
            response = client.chat.completions.create(
                model="llama-3.1-8b-instant",
                messages=[{"role": "user", "content": user_input}]
            )
            return response.choices[0].message.content

        except:
            return faq if faq else "🤖 Please provide more details."


# ============================================
# STEP 7: RUN LOOP
# ============================================

print("🚀 Banking MCP + LLM Started (type 'exit' to stop)\n")

while True:

    user_input = input("Enter your query: ")

    if user_input.lower() == "exit":
        print("👋 Exiting Banking Assistant...")
        break

    response = banking_mcp_agent(user_input)
    print(response)

🔑 Enter your Groq API Key: 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
🚀 Banking MCP + LLM Started (type 'exit' to stop)

Enter your query: how generate atm pin

🧠 Banking Agent received input: how generate atm pin
⚠️ LLM Error: Error code: 400 - {'error': {'message': 'The model `llama-3.1-70b-versatile` has been decommissioned and is no longer supported. Please refer to https://console.groq.com/docs/deprecations for a recommendation on which model to use instead.', 'type': 'invalid_request_error', 'code': 'model_decommissioned'}}
📊 Extracted → Category: transactions, Priority: medium
➡️ Decision: FAQ response only
Generating an ATM PIN (Personal Identification Number) typically requires a combination of digits that adhere to specific security guidelines. While I can provide a general outline, please note that the specific requirements may vary depending on the bank or financial institution. Here's a simplified approach to generate a PIN:

**Common guidelines for generatin

# Create a  Weather Tool MCP Server that any AI agent can use with sample use case
 Everyone,for m1 exam coding questions please go through the AST and sys libraries and try to understand how to take input in coding questions, because your code will run only when you take proper input according to the library.
 data = sys.stdin.read().strip()
lines = data.split('\n')
data = '[' + ','.join(lines) + ']'
dataset = ast.literal_eval(data)

In [ ]:
# ======================================================
# 🌦️ MCP WEATHER AGENT (ERROR-FREE VERSION)
# ======================================================

# !pip install groq requests

import requests
from groq import Groq
from google.colab import userdata


# ============================================
# 🔑 STEP 0: GROQ API (YOUR SECRET)
# ============================================

api_key = userdata.get("genapi")

if not api_key:
    raise ValueError("❌ Groq API key not found")

client = Groq(api_key=api_key)

print("✅ Groq API Connected")


# ============================================
# 🌦️ STEP 1: WEATHER TOOL (WITH FALLBACK)
# ============================================

WEATHER_API_KEY = "cf7f2e5ac0ddd67989677e6f5887731a"


def get_weather(city):
    url = "https://api.openweathermap.org/data/2.5/weather"

    params = {
        "q": city,
        "appid": WEATHER_API_KEY,
        "units": "metric"
    }

    try:
        response = requests.get(url, params=params)

        if response.status_code == 401:
            print("⚠️ Invalid API Key → Using fallback data")

            # 🔥 FALLBACK DATA (no error)
            return {
                "city": city,
                "temperature": "30",
                "condition": "clear sky (demo)",
                "humidity": "40"
            }

        if response.status_code != 200:
            return {"error": "City not found"}

        data = response.json()

        return {
            "city": city.title(),
            "temperature": data["main"]["temp"],
            "condition": data["weather"][0]["description"],
            "humidity": data["main"]["humidity"]
        }

    except:
        # 🔥 FULL FAIL SAFE
        return {
            "city": city,
            "temperature": "N/A",
            "condition": "Unavailable",
            "humidity": "N/A"
        }


# ============================================
# 🧠 STEP 2: LLM CITY EXTRACTION
# ============================================

def extract_city(user_input):
    try:
        prompt = f"""
        Extract city name from this query.
        Only return city name.

        Query: {user_input}
        """

        response = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[{"role": "user", "content": prompt}]
        )

        return response.choices[0].message.content.strip()

    except:
        # fallback
        return user_input.split()[-1]


# ============================================
# 🧠 STEP 3: DECISION ENGINE
# ============================================

def should_call_weather_tool(user_input):
    keywords = ["weather", "temperature", "rain", "forecast"]

    return any(word in user_input.lower() for word in keywords)


# ============================================
# 🤖 STEP 4: MCP AGENT
# ============================================

def weather_agent(user_input):

    print("\n🧠 Processing:", user_input)

    if should_call_weather_tool(user_input):

        print("➡️ Calling Weather Tool")

        city = extract_city(user_input)
        print("📍 City:", city)

        result = get_weather(city)

        return f"""
🌦️ Weather Report

📍 City: {result['city']}
🌡️ Temperature: {result['temperature']}°C
☁️ Condition: {result['condition']}
💧 Humidity: {result['humidity']}%
"""

    else:
        return "🤖 Ask me about weather (e.g., 'weather in Delhi')"


# ============================================
# 🚀 STEP 5: RUN
# ============================================

print("\n🌦️ MCP Weather Agent Started (type 'exit')\n")

while True:
    user_input = input("Enter query: ")

    if user_input.lower() == "exit":
        print("👋 Exiting...")
        break

    print(weather_agent(user_input))

✅ Groq API Connected

🌦️ MCP Weather Agent Started (type 'exit')

Enter query: weather in dehradun

🧠 Processing: weather in dehradun
➡️ Calling Weather Tool
📍 City: Dehradun

🌦️ Weather Report

📍 City: Dehradun
🌡️ Temperature: 27.19°C
☁️ Condition: overcast clouds
💧 Humidity: 35%

Enter query: weather in  mussoorie

🧠 Processing: weather in  mussoorie
➡️ Calling Weather Tool
📍 City: Mussoorie

🌦️ Weather Report

📍 City: Mussoorie
🌡️ Temperature: 17.42°C
☁️ Condition: overcast clouds
💧 Humidity: 40%

Enter query: weather in nepal

🧠 Processing: weather in nepal
➡️ Calling Weather Tool
📍 City: Kathmandu

🌦️ Weather Report

📍 City: Kathmandu
🌡️ Temperature: 20.12°C
☁️ Condition: scattered clouds
💧 Humidity: 60%

Enter query: exit
👋 Exiting...
